# Genetic programming — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Evolving small expression trees by error, subtree crossover, and mutation
Genetic programming is a genetic algorithm whose individuals are programs or expression trees. These cells keep the language tiny so the idea is visible: score expressions by prediction error, select better trees, alter subtrees, and watch symbolic regression improve.

In [ ]:
# 1) Score candidate expression trees by mean squared error
x=np.array([-1.,0.,1.,2.]); y=x**2+1
preds={'x+1':x+1, 'x^2+1':x**2+1, '2x+1':2*x+1}
errs={k:float(np.mean((v-y)**2)) for k,v in preds.items()}
plt.figure(figsize=(6,3)); plt.bar(list(errs.keys()), list(errs.values()), color=['tab:orange','tab:green','tab:red']); plt.ylabel('MSE')
plt.title('program fitness is output behavior, not text length')
assert errs['x^2+1']==0.0 and abs(errs['x+1']-2.0)<1e-12

In [ ]:
# 2) Subtree mutation can jump from linear to curved behavior
x=np.linspace(-2,2,100); target=x**2+1; parent=x+1; mutant=x*x+1
plt.figure(figsize=(6,3)); plt.plot(x,target,'k--',label='target'); plt.plot(x,parent,label='parent x+1'); plt.plot(x,mutant,label='mutant x^2+1')
plt.legend(); plt.title('replacing one subtree changes the function family')
assert np.mean((mutant-target)**2) < np.mean((parent-target)**2)

In [ ]:
# 3) Subtree crossover combines useful building blocks
x=np.array([-1.,0.,1.,2.]); y=x**2+x+1
left=x**2+1; right=2*x+1; child=x**2+x+1
errs=[np.mean((left-y)**2), np.mean((right-y)**2), np.mean((child-y)**2)]
plt.figure(figsize=(6,3)); plt.bar(['x^2+1','2x+1','x^2+x+1'], errs, color=['tab:blue','tab:orange','tab:green']); plt.ylabel('MSE')
plt.title('crossover assembles subexpressions')
assert abs(errs[2])<1e-12 and errs[2] < min(errs[:2])

In [ ]:
# 4) Parsimony pressure breaks ties toward simpler programs
raw_err=np.array([0.0,0.0,0.25]); sizes=np.array([5,9,3]); lam=0.05
score=raw_err+lam*sizes
plt.figure(figsize=(6,3)); plt.bar(['exact small','exact large','short imperfect'], score); plt.ylabel('penalized score')
plt.title('error plus size discourages bloated trees')
assert score[0] < score[1] and score[0] < score[2]

In [ ]:
# 5) Tiny symbolic regression loop improves the best expression
rng=np.random.default_rng(4); xs=np.linspace(-2,2,30); y=xs**2+xs+1
exprs=[('1',lambda z: np.ones_like(z)),('x',lambda z:z),('x+1',lambda z:z+1),('2x+1',lambda z:2*z+1),('x^2+1',lambda z:z*z+1),('x^2+x+1',lambda z:z*z+z+1)]
pop=rng.choice(len(exprs), size=10); best=[]
for _ in range(8):
    err=np.array([np.mean((exprs[i][1](xs)-y)**2) for i in pop]); best.append(err.min())
    probs=softmax_min(err, temp=0.5); survivors=rng.choice(pop, size=10, p=probs); pop=np.where(rng.random(10)<0.35, rng.integers(0,len(exprs),10), survivors)
plt.figure(figsize=(6,3)); plt.plot(best,'-o'); plt.xlabel('generation'); plt.ylabel('best MSE'); plt.title('program population discovers better expressions')
assert best[-1] <= best[0]